# CS224N L01 — Code Capsule: Skip-gram Softmax P(o|c)

**Waypoint**: WP03 — Word2vec Skip-gram 模型
**官方锚点**: Slides p27-30 (objective function, softmax); Notes §3.2 (Eq.4-5)

## 这段代码在看什么

这个 notebook 演示 Skip-gram 模型如何用 **softmax** 计算概率 P(o|c)——
给定中心词 c，预测它的上下文词 o 的概率分布。

核心公式（Notes Eq.4 / Slides p28）：

$$P(o|c) = rac{\exp(u_o^	op v_c)}{\sum_{w \in V} \exp(u_w^	op v_c)}$$

- **分子**: 目标词 o 和中心词 c 的相似度（点积 → 指数）
- **分母**: 全词表归一化（遍历所有 w ∈ V）—— 这就是计算瓶颈！

**为什么不能只靠文字**：Slides p28-30 给出了公式，但只有实际计算一个 mini vocabulary 的 P(o|c)，
才能理解分母归一化需要遍历全词表的计算量——这正是 WP05 负采样要解决的瓶颈。

## 1. 导入与设置

我们只需要 numpy（数值计算）和 matplotlib（可视化）。这两个包在 Colab 中预装。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 复现性种子
np.random.seed(42)

## 2. Mini Vocabulary（小词汇表）

我们用 10 个词的小词汇表，模拟 Slides p25 的 "banking" 上下文窗口示例。

每个词有**两个向量**（Slides p28）：
- $v_w$：当这个词做**中心词**时的向量（V 矩阵的行）
- $u_w$：当这个词做**上下文词**时的向量（U 矩阵的行）

注意 V ≠ U —— 同一个词的两套向量是不同的！

In [ ]:
# 10 词词汇表：金融主题 + 无关词作对比
VOCAB = [
    "banking",      # 0 - 中心词
    "crises",       # 1 - context window 词
    "into",         # 2 - context window 词
    "turning",      # 3 - context window 词
    "problems",     # 4 - context window 词
    "money",        # 5 - 语义相关词
    "economy",      # 6 - 语义相关词
    "policy",       # 7 - 语义相关词
    "cat",          # 8 - 无关词（应该得到低概率）
    "dog",          # 9 - 无关词（应该得到低概率）
]

VOCAB_SIZE = len(VOCAB)  # 10
DIM = 4  # 向量维度（小，便于透明观察）

# 中心向量矩阵 V（每行 = v_w）
V = np.array([
    [ 0.8,  0.6, -0.3,  0.1],   # banking
    [ 0.5,  0.7, -0.5,  0.2],   # crises
    [ 0.3,  0.2,  0.8, -0.1],   # into
    [ 0.4,  0.3,  0.6,  0.3],   # turning
    [ 0.6,  0.8, -0.4,  0.1],   # problems
    [ 0.7,  0.5, -0.2,  0.3],   # money
    [ 0.9,  0.4, -0.1,  0.2],   # economy
    [ 0.6,  0.3,  0.1,  0.4],   # policy
    [-0.8, -0.7,  0.5, -0.3],   # cat（无关）
    [-0.7, -0.9,  0.4, -0.2],   # dog（无关）
], dtype=np.float64)

# 上下文向量矩阵 U（每行 = u_w）—— 注意和 V 不同！
U = np.array([
    [ 0.7,  0.5, -0.2,  0.2],   # banking (as context)
    [ 0.6,  0.8, -0.6,  0.1],   # crises
    [ 0.2,  0.1,  0.7, -0.2],   # into
    [ 0.3,  0.4,  0.5,  0.2],   # turning
    [ 0.5,  0.9, -0.3,  0.0],   # problems
    [ 0.8,  0.4, -0.1,  0.3],   # money
    [ 0.9,  0.3,  0.0,  0.1],   # economy
    [ 0.5,  0.2,  0.2,  0.5],   # policy
    [-0.9, -0.6,  0.4, -0.4],   # cat
    [-0.6, -0.8,  0.3, -0.1],   # dog
], dtype=np.float64)

print(f"词汇表大小 |V| = {VOCAB_SIZE}")
print(f"向量维度 d = {DIM}")
print(f"V 矩阵形状: {V.shape}")
print(f"U 矩阵形状: {U.shape}")

## 3. Softmax 三步计算（Slides p30）

Softmax 分三步：
1. **① Dot product**: 计算 $u_w^	op v_c$（中心词和每个上下文词的相似度）
2. **② Exponentiation**: $\exp(u_w^	op v_c)$（把分数变成正数）
3. **③ Normalize**: 除以 $Z = \sum_w \exp(u_w^	op v_c)$（归一化成概率）

我们选 "banking" 作为中心词 c。

In [ ]:
# 选 banking 作为中心词（index 0）
center_idx = 0
center_word = VOCAB[center_idx]
v_c = V[center_idx]  # banking 的中心向量

print(f"中心词: '{center_word}' (index {center_idx})")
print(f"中心向量 v_{center_word} = {v_c}")
print()

# Step 1: 点积 u_w^T v_c
print("Step 1: Dot products u_w^T v_c")
print("-" * 50)
dot_products = np.zeros(VOCAB_SIZE)
for i, word in enumerate(VOCAB):
    dot_products[i] = np.dot(U[i], v_c)
    print(f"  u_{word:>10s}^T v_{center_word} = {dot_products[i]:+.6f}")

print()

# Step 2: 取指数
print("Step 2: Exponentiation exp(u_w^T v_c)")
print("-" * 50)
exp_scores = np.exp(dot_products)
for i, word in enumerate(VOCAB):
    print(f"  exp({dot_products[i]:+.6f}) = {exp_scores[i]:.6f}")

print()

# Step 3: 归一化（partition function Z）
Z = np.sum(exp_scores)
print(f"Step 3: Z = Σ exp(u_w^T v_c) = {Z:.6f}")
print(f"  （我们需要对全部 {VOCAB_SIZE} 个词计算 exp！）")
print()

# Step 4: 最终概率
probs = exp_scores / Z
print(f"Step 4: P(o|c) = exp(u_o^T v_c) / Z")
print("-" * 50)
for i, word in enumerate(VOCAB):
    bar = "█" * int(probs[i] * 50)
    print(f"  P({word:>10s} | {center_word}) = {probs[i]:.6f}  {bar}")

print(f"
  Sum = {np.sum(probs):.10f} (必须 = 1.0)")

## 运行后先看哪里

> **输出解读**：
> 1. **点积**：crises 最高（+1.15），problems 次之（+1.03），cat 最低（-1.24）。
> 2. **exp 之后**：所有值变成正数——crises = 3.158 最大，cat = 0.289 最小。
> 3. **归一化**：Z = 18.274（遍历全部 10 个词）。P(crises|banking) = 0.173 最高。
> 4. **分组观察**：context 词合计 45.8%，related 词合计 36.7%，unrelated 词仅 3.5%。

## 输出怎么解释

- **crises 概率最高**（0.173）：它的上下文向量和 banking 的中心向量点积最大（+1.15），说明在这个向量空间中它们最「兼容」。
- **cat/dog 概率极低**（<0.02）：它们的向量和 banking 几乎反向，softmax 成功把概率集中到了语义相关的词上。
- **概率分布相对平坦**：最高 0.17 vs 最低 0.016——这是因为 toy 向量还没经过充分训练。真实训练后的分布会更尖锐。

## 4. 概率分布可视化

绿色 = context window 词，蓝色 = 语义相关词，红色 = 无关词。

In [ ]:
# 概率分布柱状图
context_idx = [1, 2, 3, 4]  # crises, into, turning, problems
related_idx = [5, 6, 7]     # money, economy, policy
unrelated_idx = [8, 9]      # cat, dog

colors = ['#2ecc71' if i in context_idx else 
          '#3498db' if i in related_idx else 
          '#e74c3c' for i in range(VOCAB_SIZE)]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(VOCAB_SIZE), probs, color=colors, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Output word w', fontsize=12)
ax.set_ylabel('P(w | "banking")', fontsize=12)
ax.set_title('Skip-gram Softmax: P(o|c) for center word "banking"
'
             '(Green=context window, Blue=semantically related, Red=unrelated)', 
             fontsize=13, fontweight='bold')
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels(VOCAB, rotation=45, ha='right')

# 在柱子上标注概率值
for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
            f'{prob:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Context window'),
                   Patch(facecolor='#3498db', label='Semantically related'),
                   Patch(facecolor='#e74c3c', label='Unrelated')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('outputs/skipgram-softmax-prob-distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/skipgram-softmax-prob-distribution.png")

## 5. 点积热力图

每个格子 = $u_o^	op v_c$（行 o 的上下文向量 · 列 c 的中心向量）。
颜色越暖 = 点积越高 → P(o|c) 越大。

In [ ]:
# 全词表点积矩阵
full_dots = U @ V.T

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(full_dots, cmap='RdYlBu_r', aspect='auto', vmin=-1.5, vmax=1.5)
ax.set_xticks(range(VOCAB_SIZE))
ax.set_xticklabels(VOCAB, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(VOCAB_SIZE))
ax.set_yticklabels(VOCAB, fontsize=9)
ax.set_xlabel('Center word c (v_c)', fontsize=11)
ax.set_ylabel('Context word o (u_o)', fontsize=11)
ax.set_title('Dot Products u_o^T v_c for All (center, context) Pairs', 
             fontsize=12, fontweight='bold')

for i in range(VOCAB_SIZE):
    for j in range(VOCAB_SIZE):
        val = full_dots[i, j]
        color = 'white' if abs(val) > 0.8 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='u_o^T v_c', shrink=0.8)
plt.tight_layout()
plt.savefig('outputs/skipgram-softmax-dot-product-heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/skipgram-softmax-dot-product-heatmap.png")

## 6. 分母（Partition Function）的计算代价

分母 $Z = \sum_{w \in V} \exp(u_w^	op v_c)$ 需要遍历**整个词汇表**。

真实 word2vec 的 $|V| pprox 50{,}000	ext{-}100{,}000$，每步训练都要算这么多次 exp()！

这就是 WP05 负采样要解决的问题——用 k 个负样本的二分类替代全词表 softmax。

In [ ]:
# Partition function 计算代价可视化
vocab_sizes = [10, 100, 1000, 10000, 50000, 100000]
time_per_exp_us = 1.0  # 假设每次 exp 需要 1 微秒
total_time_ms = [v * time_per_exp_us / 1000 for v in vocab_sizes]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(vocab_sizes)), total_time_ms, color='#9b59b6', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(vocab_sizes)))
ax.set_xticklabels([f'{v:,}' for v in vocab_sizes])
ax.set_xlabel('Vocabulary size |V|', fontsize=12)
ax.set_ylabel('Time for one softmax (ms)', fontsize=12)
ax.set_title('Partition Function Cost: O(|V|) per Training Step',
             fontsize=12, fontweight='bold')

for i, (v, t) in enumerate(zip(vocab_sizes, total_time_ms)):
    ax.text(i, t + 0.5, f'{t:.1f} ms', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.annotate('Real word2vec
|V| ≈ 50K-100K', 
            xy=(4, total_time_ms[4]), xytext=(3, total_time_ms[4] + 15),
            fontsize=10, ha='center',
            arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/skipgram-softmax-partition-cost.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/skipgram-softmax-partition-cost.png")

## 7. 交叉熵损失（Notes Eq.5）

训练目标是最小化交叉熵：

$$\min_{U,V} \; \mathbb{E}_{o,c}\left[-\log p_{U,V}(o|c)ight]$$

对于单个观察到的 (center, context) 对，损失就是 $-\log P(o|c)$。

假设实际观察到的上下文词是 "problems"（在 banking 的窗口里）：

In [ ]:
# 交叉熵损失计算
observed_idx = 4  # "problems"
observed_word = VOCAB[observed_idx]
loss = -np.log(probs[observed_idx])

print(f"Observed context word: '{observed_word}' (index {observed_idx})")
print(f"P('{observed_word}' | '{center_word}') = {probs[observed_idx]:.6f}")
print(f"L = -log P('{observed_word}' | '{center_word}') = -log({probs[observed_idx]:.6f}) = {loss:.6f}")
print()
print("这个损失会反向传播，更新 U 和 V 中相关的向量，")
print("使 P('problems' | 'banking') 在下一步变得更大。")

## 和本讲哪个 waypoint 对应

本 notebook 对应 **WP03**（Skip-gram 模型 + softmax 概率计算）。

它直接演示了：
- Slides p28-30 的 softmax 三步（dot product → exponentiation → normalize）
- Notes §3.2 Eq.4（softmax 概率公式）和 Eq.5（交叉熵损失）

分母 O(|V|) 的计算瓶颈自然引出 **WP05** 的负采样替代方案。

## 容易误解的地方

> **⚠️ 常见误区**
> 
> 1. **每个词只有一个向量？** 不是。Skip-gram 中每个词有**两个**向量：$v_c$（当它是中心词时）和 $u_w$（当它是上下文词时）。本 notebook 的 V 和 U 矩阵就是这两套向量。
> 
> 2. **点积 = 概率？** 不是。点积可以是负数，概率必须在 [0,1]。softmax 的作用就是把任意实数分数变成合法概率分布。
> 
> 3. **概率最高的词一定是中心词自己？** 不一定。在这个例子中，crises 的 P(o|c) 比 banking 自己还高——因为 crises 的上下文向量和 banking 的中心向量更对齐。
> 
> 4. **Z 很小所以计算便宜？** 本例 |V|=10 所以 Z 只需加 10 项。真实词表 |V|≈50K-100K，每步训练需要 50K+ 次 exp()——这就是负采样的动机。

## 总结

| 概念 | 公式/值 |
|------|---------|
| Softmax 概率 | $P(o\|c) = \exp(u_o^	op v_c) / \sum_w \exp(u_w^	op v_c)$ |
| 每词两向量 | $v_w$ (center), $u_w$ (context) |
| 分母代价 | $O(\|V\|)$ — 每步遍历全词表 |
| 交叉熵损失 | $-\log P(o\|c)$ |
| 本例 Z | 18.273793（10 个词的 exp 之和）|
| 本例最高概率 | P(crises\|banking) = 0.172826 |
| 本例损失 | L = 1.875468（对 observed word "problems"）|

**关键洞察**: 分母需要遍历整个词汇表 → 这就是为什么负采样 (WP05) 用 logistic 二分类替代全 softmax。